<a href="https://colab.research.google.com/github/Siddtiwa/Karpathy-assignments/blob/main/Makemore2/5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import torch
import torch.nn.functional as F

In [2]:
uploaded = files.upload()

Saving names.txt to names.txt


In [37]:
words = open("names.txt", "r").read().splitlines()

In [38]:
vocab = sorted(list(set("".join(words))))
len(vocab)

26

In [39]:
itos = {i+1:c for i, c in enumerate(vocab)}
itos[0] = "."
stoi = {c:i for i, c in itos.items()}

In [40]:
stoi

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [41]:
itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [42]:
def data_processing(words):
  x,y = [],[]
  context_size = 3

  for word in words:
     context = [0] * context_size
     for ch in word + ".":
      ix = stoi[ch]
      x.append(context)
      y.append(ix)
      context = context[1:] + [ix]

  x = torch.tensor(x)
  y = torch.tensor(y)

  return x, y

In [43]:
x,y = data_processing(words)

In [82]:
xtrain = x[:int(0.8*len(x))]
ytrain = y[:int(0.8*len(y))]
xdev = x[int(0.8*len(x)):int(0.9*len(x))]
ydev = y[int(0.8*len(y)):int(0.9*len(y))]
xtest = x[int(0.9*len(x)):]
ytest = y[int(0.9*len(y)):]

In [72]:
g = torch.Generator().manual_seed(2147483647)
c = torch.randn((27,10), generator = g)
w1 = torch.randn((30,200), generator = g)
b1 = torch.randn((200), generator = g)
w2 = torch.randn((200,27), generator = g)
b2 = torch.randn((27), generator = g)
parameters = [w1,b1,w2,b2]
for p in parameters:
  p.requires_grad = True

In [87]:
g = torch.Generator().manual_seed(2147483647)
for i in range(200000):
  ix = torch.randint(0, xtrain.shape[0], (100,))

  emb = c[xtrain[ix]]
  h = torch.tanh(emb.view(-1,30) @ w1 + b1)
  logits = h @ w2 + b2
  loss = F.cross_entropy(logits, ytrain[ix])
  print(loss)

  for p in parameters:
    p.grad = None
  loss.backward()

  lr = 0.1 if i < 100000 else 0.001
  for p in parameters:
    p.data += -lr * p.grad

Streaming output truncated to the last 5000 lines.
tensor(1.8241, grad_fn=<NllLossBackward0>)
tensor(1.9656, grad_fn=<NllLossBackward0>)
tensor(1.8937, grad_fn=<NllLossBackward0>)
tensor(1.7530, grad_fn=<NllLossBackward0>)
tensor(1.8970, grad_fn=<NllLossBackward0>)
tensor(1.9473, grad_fn=<NllLossBackward0>)
tensor(1.9753, grad_fn=<NllLossBackward0>)
tensor(1.9541, grad_fn=<NllLossBackward0>)
tensor(2.0804, grad_fn=<NllLossBackward0>)
tensor(2.0339, grad_fn=<NllLossBackward0>)
tensor(1.8553, grad_fn=<NllLossBackward0>)
tensor(1.9144, grad_fn=<NllLossBackward0>)
tensor(2.1446, grad_fn=<NllLossBackward0>)
tensor(1.9505, grad_fn=<NllLossBackward0>)
tensor(1.9348, grad_fn=<NllLossBackward0>)
tensor(1.9677, grad_fn=<NllLossBackward0>)
tensor(1.9052, grad_fn=<NllLossBackward0>)
tensor(2.0958, grad_fn=<NllLossBackward0>)
tensor(1.7910, grad_fn=<NllLossBackward0>)
tensor(2.2163, grad_fn=<NllLossBackward0>)
tensor(1.9986, grad_fn=<NllLossBackward0>)
tensor(1.8999, grad_fn=<NllLossBackward0>)
ten

In [88]:
emb = c[xdev]
h = torch.tanh(emb.view(-1,30) @ w1 + b1)
logits = h @ w2 + b2
loss = F.cross_entropy(logits, ydev)
loss

tensor(2.3724, grad_fn=<NllLossBackward0>)

In [89]:
emb = c[xtest]
h = torch.tanh(emb.view(-1,30) @ w1 + b1)
logits = h @ w2 + b2
loss = F.cross_entropy(logits, ytest)
loss

tensor(2.4226, grad_fn=<NllLossBackward0>)

In [77]:
g = torch.Generator().manual_seed(2147483647)

for _ in range(20):
  out = []
  context = [0] * 3
  while True:
    emb = c[torch.tensor([context])]
    h = torch.tanh(emb.view(1,-1) @ w1 +b1)
    logits = h @ w2 + b2
    probs = F.softmax(logits, dim=1)
    ix = torch.multinomial(probs, num_samples=1, generator=g).item()
    context = context[1:] + [ix]
    out.append(ix)
    if ix == 0:
      break
  print("".join(itos[i] for i in out))

dex.
maleah.
makilah.
tah.
mellimitta.
mella.
kaman.
arrelicausttlhi.
gotti.
molie.
luvu.
jerred.
jenter.
masede.
eliaviyn.
zyonstihani.
gavtahlas.
dasord.
anell.
gyan.


In [81]:
ytrain.shape, ydev.shape, ytest.shape

(torch.Size([182516]), torch.Size([22815]), torch.Size([22815, 3]))